In [11]:
# Imports
import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd
import numpy as np

### 1) Directory Structure

In [8]:
load_dotenv()

api_key = os.getenv("API_KEY")
repo_owner = 'bedbad'
repo_name = 'justpyplot'

url = "https://api.github.com/graphql"
headers = {
    "Authorization": f"Bearer {api_key}"
}

# Rest of code in this cell is partly AI generated
def get_folder_contents(expression="HEAD:"):
    """Fetches the contents of a specific folder path."""
    query = """
    query($owner: String!, $name: String!, $expression: String!) {
      repository(owner: $owner, name: $name) {
        object(expression: $expression) {
          ... on Tree {
            entries {
              name
              path
              type
              extension
            }
          }
        }
      }
    }
    """
    variables = {
        "owner": repo_owner, 
        "name": repo_name, 
        "expression": expression
    }
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    return response.json()

all_items = []
# Start at the root of the repository
folders_to_check = ["HEAD:"]

while folders_to_check:
    current_folder = folders_to_check.pop(0)
    data = get_folder_contents(current_folder)
    try:
        entries = data['data']['repository']['object']['entries']
    except (KeyError, TypeError):
        continue
        
    for entry in entries:
        all_items.append({
            'Name': entry['name'],
            'Path': entry['path'],
            'Type': entry['type'], # 'blob' for files, 'tree' for folders
            'Extension': entry.get('extension', '')
        })
        
        # If the item is a folder (tree), queue it up to fetch its contents next
        if entry['type'] == 'tree':
            folders_to_check.append(f"HEAD:{entry['path']}/")

# Dump a sample to json for reference
with open("directory_query.json", "w") as file:
    json.dump(all_items[:100], file, indent=4)
    
print(f"Found {len(all_items)} total files and folders.")

Found 38 total files and folders.


In [18]:
df = pd.DataFrame(all_items)

# Change type column to something human readible
df['Type'] = np.where(df['Type'] == 'tree', 'folder', 'file')

# Sort alphabetically by path
df_sorted = df.sort_values(by='Path').reset_index(drop=True)

# Create CSV file
df_sorted.to_csv("repo_directory_structure.csv", index=False)

print("Created CSV File")
df_sorted.head(15)

Created CSV File


,Name,Path,Type,Extension
0,.github,.github,folder,
1,funding.yaml,.github/funding.yaml,file,.yaml
2,workflows,.github/workflows,folder,
3,python-publish.yml,.github/workflows/python-publish.yml,file,.yml
4,.gitignore,.gitignore,file,
5,.readthedocs.yaml,.readthedocs.yaml,file,.yaml
6,CONTRIBUTING.md,CONTRIBUTING.md,file,.md
7,LICENSE,LICENSE,file,
8,README.md,README.md,file,.md
9,docs,docs,folder,
